In [1]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Load pre-trained Faster R-CNN model
def get_faster_rcnn(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

    # Replace the classifier with a new one for your number of classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

# Define number of classes (including background)
num_classes = 2  # 1 for two-wheeler, 1 for background
model = get_faster_rcnn(num_classes)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
import os
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import json

class TwoWheelerDataset(torch.utils.data.Dataset):
    def __init__(self, root, annotation_file, transforms=None):
        self.root = root
        self.transforms = transforms
        self.annotations = json.load(open(annotation_file))

        self.imgs = {img["id"]: img["file_name"] for img in self.annotations["images"]}
        self.annotations = {ann["image_id"]: ann for ann in self.annotations["annotations"]}

    def __getitem__(self, idx):
        img_id = list(self.imgs.keys())[idx]
        img_path = os.path.join(self.root, self.imgs[img_id])
        img = Image.open(img_path).convert("RGB")

        # Get bounding box and labels
        ann = self.annotations[img_id]
        boxes = [ann["bbox"]]  # Convert to tensor format
        labels = torch.tensor([1], dtype=torch.int64)  # Two-wheeler class label

        # Convert bounding boxes to required format
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [3]:
def fix_bboxes(bboxes):
    """
    Fix bounding boxes in (x, y, width, height) format and ensure valid dimensions.
    
    Args:
    - bboxes: List of bounding boxes [x, y, width, height].
    
    Returns:
    - Fixed bounding boxes in [x1, y1, x2, y2] format.
    """
    fixed_bboxes = []
    
    for box in bboxes:
        x, y, w, h = box  # Original format
        
        # Ensure width and height are positive
        if w <= 0 or h <= 0:
            print(f"Warning: Invalid bbox {box} skipped")
            continue

        x1, y1, x2, y2 = x, y, x + w, y + h  # Convert to [x1, y1, x2, y2] format
        fixed_bboxes.append([x1, y1, x2, y2])
    
    return fixed_bboxes


In [4]:
def remove_invalid_boxes(bboxes, labels, min_size=1):
    """
    Remove bounding boxes that have zero or near-zero width/height.
    
    Args:
    - bboxes: List of bounding boxes [x1, y1, x2, y2].
    - labels: Corresponding labels.
    - min_size: Minimum allowed width/height.
    
    Returns:
    - Filtered bounding boxes and labels.
    """
    valid_boxes = []
    valid_labels = []
    
    for i, box in enumerate(bboxes):
        x1, y1, x2, y2 = box
        width, height = x2 - x1, y2 - y1
        
        if width >= min_size and height >= min_size:
            valid_boxes.append([x1, y1, x2, y2])
            valid_labels.append(labels[i])  # Keep label if box is valid
        else:
            print(f"Warning: Removed invalid bbox {box}")
    
    return valid_boxes, valid_labels


In [5]:
import torch
torch.cuda.empty_cache()

In [6]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    print(f"Available GPU(s): {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
    # Set memory growth (similar to TensorFlow's `set_memory_growth`)
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True  # Optimize performance for fixed-size inputs
    torch.backends.cudnn.enabled = True  # Enable cuDNN

else:
    print("CUDA is not available.")


Available GPU(s): 1
GPU 0: NVIDIA GeForce GTX 970


In [14]:
from torch.utils.data import DataLoader

# Define transformations
transform = T.Compose([T.ToTensor()])

# Load dataset
dataset = TwoWheelerDataset("Data_twowheeler/train", "Data_twowheeler/train/_two_wheeler_annotations_train.json", transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    
    for images, targets in dataloader:
        images = list(img.to(device) for img in images)
        
        # Fix bounding boxes before sending to model
        fixed_targets = []
        for target in targets:
            fixed_boxes = fix_bboxes(target["boxes"])  # Convert [x, y, w, h] → [x1, y1, x2, y2]
            valid_boxes, valid_labels = remove_invalid_boxes(fixed_boxes, target["labels"])  # Remove invalid boxes
            
            # Recreate the target dictionary
            fixed_target = {
                "boxes": torch.tensor(valid_boxes, dtype=torch.float32).to(device),
                "labels": torch.tensor(valid_labels, dtype=torch.int64).to(device)
            }
            fixed_targets.append(fixed_target)

        optimizer.zero_grad()
        loss_dict = model(images, fixed_targets)  # Pass cleaned targets
        loss = sum(loss for loss in loss_dict.values())

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

print("Training complete!") 

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.94 GiB of which 4.31 MiB is free. Process 47653 has 3.46 GiB memory in use. Including non-PyTorch memory, this process has 473.62 MiB memory in use. Of the allocated memory 381.24 MiB is allocated by PyTorch, and 28.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch

def compute_iou(box1, box2):
    """
    Compute Intersection over Union (IoU) between two bounding boxes.
    
    Args:
    - box1, box2: Bounding boxes in [x1, y1, x2, y2] format.
    
    Returns:
    - IoU score
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    # Compute intersection
    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    # Compute union
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = box1_area + box2_area - intersection

    if union == 0:
        return 0.0  # Avoid division by zero

    return intersection / union

In [ ]:
def mean_average_precision(pred_boxes, gt_boxes, iou_threshold=0.5):
    """
    Compute mean Average Precision (mAP) for object detection.
    
    Args:
    - pred_boxes: List of predicted boxes (x1, y1, x2, y2).
    - gt_boxes: List of ground-truth boxes (x1, y1, x2, y2).
    - iou_threshold: IoU threshold to count a true positive.
    
    Returns:
    - mAP score
    """
    tp, fp, fn = 0, 0, 0

    for gt in gt_boxes:
        ious = [compute_iou(gt, pred) for pred in pred_boxes]
        if len(ious) > 0 and max(ious) >= iou_threshold:
            tp += 1  # True positive
        else:
            fn += 1  # False negative

    for pred in pred_boxes:
        ious = [compute_iou(pred, gt) for gt in gt_boxes]
        if len(ious) == 0 or max(ious) < iou_threshold:
            fp += 1  # False positive

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return {"Precision": precision, "Recall": recall, "mAP": (precision + recall) / 2}
